In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 


In [3]:
books = pd.read_csv(r'raw\Books.csv')
rating = pd.read_csv(r'raw\Ratings.csv')
users = pd.read_csv(r'raw\Users.csv')

C:\Users\Eyeden\AppData\Local\Temp\ipykernel_3848\3168762951.py:1: DtypeWarning: Columns (0: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv(r'raw\Books.csv')


In [4]:
rating.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [5]:
users.head()

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [6]:
rating.head()

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [7]:
datasets = [("Books",books), ("Users",users), ("Rating",rating)]
for name,df in datasets:
    print(f"Datasets: {name}")
    print(df.shape)
    print("\n")

Datasets: Books
(271360, 8)


Datasets: Users
(278858, 3)


Datasets: Rating
(1149780, 3)




## missing values

In [8]:
datasets = [("Books",books), ("Users",users), ("Rating",rating)]
for name,dataset in datasets:
    print(f"Dataset : {name}")
    print(dataset.isnull().sum())
    print("\n")

Dataset : Books
ISBN                   0
Book-Title             0
Book-Author            2
Year-Of-Publication    0
Publisher              2
Image-URL-S            0
Image-URL-M            0
Image-URL-L            3
dtype: int64


Dataset : Users
User-ID          0
Location         0
Age         110762
dtype: int64


Dataset : Rating
User-ID        0
ISBN           0
Book-Rating    0
dtype: int64




## Duplicates

In [9]:
datasets = [("Books",books), ("Users",users), ("Rating",rating)]
for name,dataset in datasets:
    print("Dataset : ",{name})
    print(dataset.duplicated().sum())
    print("\n")

Dataset :  {'Books'}
0


Dataset :  {'Users'}
0


Dataset :  {'Rating'}
0




# Popularity Based Recommender System

We only display the top 50 books which have the highest ratings and books which have more than 200 ratings 

In [10]:
rating_with_name = rating.merge(books,on='ISBN')

In [11]:
num_rating = rating_with_name.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating.rename(columns={'Book-Rating':'num_rating'},inplace=True)
num_rating

,Book-Title,num_rating
0,A Light in the Storm: The Civil War Diary of ...,4
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1
4,Beyond IBM: Leadership Marketing and Finance ...,1
...,...,...
241066,Ã?Â?lpiraten.,2
241067,Ã?Â?rger mit Produkt X. Roman.,4
241068,Ã?Â?sterlich leben.,1
241069,Ã?Â?stlich der Berge.,3


In [12]:
avg_rating = rating_with_name.groupby('Book-Title')['Book-Rating'].mean().reset_index()
avg_rating.rename(columns={'Book-Rating':'avg_rating'},inplace=True)
avg_rating

,Book-Title,avg_rating
0,A Light in the Storm: The Civil War Diary of ...,2.250000
1,Always Have Popsicles,0.000000
2,Apple Magic (The Collector's series),0.000000
3,"Ask Lily (Young Women of Faith: Lily Series, ...",8.000000
4,Beyond IBM: Leadership Marketing and Finance ...,0.000000
...,...,...
241066,Ã?Â?lpiraten.,0.000000
241067,Ã?Â?rger mit Produkt X. Roman.,5.250000
241068,Ã?Â?sterlich leben.,7.000000
241069,Ã?Â?stlich der Berge.,2.666667


In [13]:
popularity_df = num_rating.merge(avg_rating,on='Book-Title')
popularity_df

,Book-Title,num_rating,avg_rating
0,A Light in the Storm: The Civil War Diary of ...,4,2.250000
1,Always Have Popsicles,1,0.000000
2,Apple Magic (The Collector's series),1,0.000000
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1,8.000000
4,Beyond IBM: Leadership Marketing and Finance ...,1,0.000000
...,...,...,...
241066,Ã?Â?lpiraten.,2,0.000000
241067,Ã?Â?rger mit Produkt X. Roman.,4,5.250000
241068,Ã?Â?sterlich leben.,1,7.000000
241069,Ã?Â?stlich der Berge.,3,2.666667


In [14]:
popularity_df=popularity_df[popularity_df['num_rating']>=250].sort_values('avg_rating',ascending=False).head(50).reset_index(drop=True) 

In [15]:
popular_df = popularity_df.merge(books,on='Book-Title').drop_duplicates('Book-Title')[['Book-Title','Book-Author','Image-URL-M','num_rating','avg_rating']].reset_index(drop=True)
popular_df.to_csv(r"cleaned\popular_books.csv")

In [16]:
len(popular_df)

50

In [17]:
import pickle
pickle.dump(popular_df,open('model\\popular.pkl','wb'))

## Collaborative Filtering Based Recommender System

In [18]:
x = rating_with_name.groupby('User-ID')['Book-Rating'].count() >200
experienced_users= x[x].index

In [19]:
#based on User
filtered_rating = rating_with_name[rating_with_name['User-ID'].isin(experienced_users)]

In [20]:
#based on user
y=filtered_rating.groupby('Book-Title')['Book-Rating'].count() >=50
famous_books = y[y].index

In [21]:
final_ratings = filtered_rating[filtered_rating['Book-Title'].isin(famous_books)]

In [22]:
pt = final_ratings.pivot_table(index='Book-Title',columns='User-ID',values='Book-Rating')

In [23]:
pt.fillna(0,inplace=True)

User-ID,254,2276,2766,2977,3363,4017,4385,6251,6323,6543,...,271705,273979,274004,274061,274301,274308,275970,277427,277639,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Bend in the Road,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Model Buildiing

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
similarity_scores = cosine_similarity(pt)

In [26]:
def recommend(book_name):
    #fetch index from book name
    index = np.where(pt.index==book_name)[0][0]
    distances = similarity_scores[index]
    similar_items = sorted(list(enumerate(similarity_scores[index])),key=lambda x:x[1],reverse=True)[1:6]
    data =[]
    for i in similar_items:
        item=[]
        temp_df = books[books['Book-Title']==pt.index[i[0]]]
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
        item.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))

        data.append(item)
    return data


In [27]:
# Test
recommend("1984")

[['Animal Farm',
  'George Orwell',
  'http://images.amazon.com/images/P/0451526341.01.MZZZZZZZ.jpg'],
 ["The Handmaid's Tale",
  'Margaret Atwood',
  'http://images.amazon.com/images/P/0449212602.01.MZZZZZZZ.jpg'],
 ['Brave New World',
  'Aldous Huxley',
  'http://images.amazon.com/images/P/0060809833.01.MZZZZZZZ.jpg'],
 ['The Vampire Lestat (Vampire Chronicles, Book II)',
  'ANNE RICE',
  'http://images.amazon.com/images/P/0345313860.01.MZZZZZZZ.jpg'],
 ['The Hours : A Novel',
  'Michael Cunningham',
  'http://images.amazon.com/images/P/0312243022.01.MZZZZZZZ.jpg']]

In [28]:
pickle.dump(pt,open(r'model\pt.pkl','wb'))
pickle.dump(books,open(r'model\books.pkl','wb'))
pickle.dump(similarity_scores,open(r'model\similarity_scores.pkl','wb'))